# Task 1 - SQL in R Analytics
### CP60056E Databases and Analytics - NorthStar Case Study

> Change runtime to **R**: `Runtime -> Change runtime type -> R`

In [ ]:
install.packages(c('sqldf','dplyr','tools'), quiet=TRUE)
library(sqldf); library(dplyr); library(tools)
cat('Packages loaded.\n')

Warning message:
“package ‘tools’ is a base package, and should not be updated”
also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




Packages loaded.


In [ ]:
DATA_PATH <- '/content/'
customers  <- read.csv(paste0(DATA_PATH,'customers.csv'),  stringsAsFactors=FALSE)
orders     <- read.csv(paste0(DATA_PATH,'orders.csv'),     stringsAsFactors=FALSE)
deliveries <- read.csv(paste0(DATA_PATH,'deliveries.csv'), stringsAsFactors=FALSE)
complaints <- read.csv(paste0(DATA_PATH,'complaints.csv'), stringsAsFactors=FALSE)
drivers    <- read.csv(paste0(DATA_PATH,'drivers.csv'),    stringsAsFactors=FALSE)
hubs       <- read.csv(paste0(DATA_PATH,'hubs.csv'),       stringsAsFactors=FALSE)
vehicles   <- read.csv(paste0(DATA_PATH,'vehicles.csv'),   stringsAsFactors=FALSE)
incidents  <- read.csv(paste0(DATA_PATH,'incidents.csv'),  stringsAsFactors=FALSE)
for (df_name in c('orders','deliveries','customers','drivers','vehicles')) {
  df <- get(df_name)
  for (col in grep('zone',names(df),value=TRUE,ignore.case=TRUE))
    df[[col]] <- tools::toTitleCase(tolower(trimws(df[[col]])))
  assign(df_name, df)
}
cat(sprintf('Loaded: %d orders, %d deliveries, %d complaints\n', nrow(orders), nrow(deliveries), nrow(complaints)))

Loaded: 1250 orders, 950 deliveries, 320 complaints


## Query 1 - Delivery Failure Rate by Zone

Joins deliveries and orders. CASE WHEN performs conditional aggregation in a single pass - more efficient than multiple subqueries.

In [ ]:
q1 <- sqldf("
  SELECT o.pickup_zone AS zone,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
    SUM(CASE WHEN d.delivery_status = 'Late'   THEN 1 ELSE 0 END) AS late,
    SUM(CASE WHEN d.delivery_status = 'OnTime' THEN 1 ELSE 0 END) AS on_time,
    ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating,
    ROUND(AVG(d.manual_route_override_count),   2) AS avg_overrides,
    ROUND(AVG(d.fuel_or_charge_cost),           2) AS avg_cost
  FROM deliveries d JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone ORDER BY failed DESC
")
q1$failure_rate_pct <- round(q1$failed / q1$total_deliveries * 100, 1)
print(q1)

       zone total_deliveries failed late on_time avg_rating avg_overrides
1     North              135     22    0      92       3.90          0.70
2   Central              110     22    0      61       3.62          1.62
3      East              156     19    0     106       3.91          0.79
4 Riverside              119     18    0      76       3.86          0.73
5      West              114     14    0      79       3.90          0.81
6     South              139     14    0     103       4.05          0.69
7   Airport              113     12    0      70       3.98          1.81
8       Ctr               64     11    0      29       3.43          0.73
  avg_cost failure_rate_pct
1    12.07             16.3
2    12.15             20.0
3    12.57             12.2
4    12.39             15.1
5    11.94             12.3
6    12.48             10.1
7    17.08             10.6
8    12.07             17.2


## Query 2 - High-Complaint Customers

Three-table JOIN with HAVING clause. Connects complaints and order history into a single customer view.

In [ ]:
q2 <- sqldf("
  SELECT c.customer_id, c.home_zone, c.customer_type, c.loyalty_score,
    COUNT(DISTINCT cp.complaint_id) AS complaint_count,
    SUM(cp.compensation_amount)     AS total_compensation,
    ROUND(AVG(o.order_value), 2)    AS avg_order_value,
    COUNT(DISTINCT o.order_id)      AS total_orders
  FROM customers c
  JOIN complaints cp ON c.customer_id = cp.customer_id
  JOIN orders     o  ON c.customer_id = o.customer_id
  GROUP BY c.customer_id, c.home_zone, c.customer_type, c.loyalty_score
  HAVING complaint_count >= 2
  ORDER BY complaint_count DESC, total_compensation DESC
  LIMIT 15
")
print(q2)

   customer_id home_zone customer_type loyalty_score complaint_count
1        C0368     North      Consumer          49.5               4
2        C0545     South      Consumer          66.9               3
3        C0372      West      Consumer          26.2               3
4        C0242      East      Consumer          83.8               3
5        C0421   Central      Consumer          59.0               3
6        C0573   Airport           SME          57.3               3
7        C0172     North      Consumer          75.4               3
8        C0282 Riverside      Consumer          71.4               3
9        C0110      East      Consumer            NA               3
10       C0191     North      Consumer          58.9               3
11       C0142     South      Consumer          47.0               3
12       C0626     South      Consumer          61.6               3
13       C0351   Central    Enterprise          48.6               2
14       C0078      East      Cons

## Query 3 - Drivers with Above-Average Failure Rate

Uses a correlated subquery in HAVING to dynamically compute fleet-wide average failure rate.

In [ ]:
q3 <- sqldf("
  SELECT dr.driver_id, dr.base_zone, dr.employment_type,
    dr.training_score, dr.driver_rating, dr.years_experience,
    COUNT(d.delivery_id) AS total_jobs,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_jobs,
    ROUND(CAST(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS FLOAT)
          / COUNT(d.delivery_id) * 100, 1) AS fail_rate_pct,
    ROUND(AVG(d.manual_route_override_count), 2) AS avg_overrides
  FROM drivers dr JOIN deliveries d ON dr.driver_id = d.driver_id
  GROUP BY dr.driver_id, dr.base_zone, dr.employment_type,
           dr.training_score, dr.driver_rating, dr.years_experience
  HAVING fail_rate_pct > (
    SELECT ROUND(CAST(SUM(CASE WHEN d2.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS FLOAT)
                 / COUNT(d2.delivery_id) * 100, 1)
    FROM deliveries d2
  )
  ORDER BY fail_rate_pct DESC LIMIT 12
")
print(q3)

   driver_id base_zone employment_type training_score driver_rating
1       D051      West        FullTime           75.4          3.58
2       D063     North        PartTime           85.7          4.03
3       D092      East        FullTime           88.2          4.24
4       D104      West        FullTime           87.7          3.45
5       D024 Riverside        PartTime           71.4          3.35
6       D103   Central        FullTime           72.5          4.40
7       D111   Airport        FullTime           79.2          4.12
8       D132     South        Contract           77.6          4.20
9       D147      West        FullTime           66.4          3.80
10      D170      West        FullTime           75.2          4.48
11      D010      West        FullTime           70.0          3.95
12      D005     North        FullTime           69.7          4.14
   years_experience total_jobs failed_jobs fail_rate_pct avg_overrides
1                 3          2           2   

## Query 4 - Vehicle Incident Analysis

LEFT JOIN preserves all vehicles. Ordered by incident count then battery health ascending (most critical first).

In [ ]:
q4 <- sqldf("
  SELECT v.vehicle_id, v.vehicle_type, v.assigned_zone,
    v.battery_health_pct, v.maintenance_status,
    COUNT(DISTINCT i.incident_id) AS incident_count,
    SUM(CASE WHEN i.severity = 'High' THEN 1 ELSE 0 END) AS high_severity,
    ROUND(AVG(i.resolved_hours), 1) AS avg_resolution_hrs,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_fuel_cost
  FROM vehicles v
  JOIN deliveries d ON v.vehicle_id = d.vehicle_id
  LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
  GROUP BY v.vehicle_id, v.vehicle_type, v.assigned_zone,
           v.battery_health_pct, v.maintenance_status
  HAVING incident_count > 0
  ORDER BY incident_count DESC, battery_health_pct ASC LIMIT 15
")
print(q4)

   vehicle_id vehicle_type assigned_zone battery_health_pct maintenance_status
1        V047           EV           Ctr               93.7          Scheduled
2        V108       Diesel       Airport               54.6           InRepair
3        V030     CargoVan         North               78.0             Active
4        V097           EV           Ctr               92.1             Active
5        V046           EV         North               95.8             Active
6        V005     CargoVan          West               58.6             Active
7        V076       Diesel       Central               65.8           InRepair
8        V009     CargoVan         South               68.8             Active
9        V088       Diesel         North               80.3           InRepair
10       V042           EV          East               80.5           InRepair
11       V035     CargoVan          East               83.6             Active
12       V025       Diesel       Airport            

## Query 5 - Hub Performance Overview

Aggregates delivery and complaint data per hub to compare operational efficiency against hub capacity scores.

In [ ]:
q5 <- sqldf("
  SELECT h.hub_name, h.zone, h.hub_type, h.capacity_score,
    COUNT(DISTINCT d.delivery_id) AS deliveries_processed,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failures,
    COUNT(DISTINCT cp.complaint_id) AS complaints_linked,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_op_cost
  FROM hubs h
  JOIN deliveries d ON h.hub_id = d.hub_id
  LEFT JOIN orders     o  ON d.order_id   = o.order_id
  LEFT JOIN complaints cp ON o.customer_id = cp.customer_id
  GROUP BY h.hub_id, h.hub_name, h.zone, h.hub_type, h.capacity_score
  ORDER BY failures DESC
")
print(q5)

        hub_name      zone  hub_type capacity_score deliveries_processed
1  Midtown Relay   Central  Charging             63                  128
2   Central Core   Central   Control             88                  115
3      West Gate      West  Dispatch             69                  127
4    Airport Hub   Airport  Dispatch             71                  104
5 North Exchange     North  Dispatch             82                  136
6  Riverside Hub Riverside Warehouse             66                  115
7     South Link     South  Dispatch             78                  106
8      East Dock      East Warehouse             74                  119
  failures complaints_linked avg_op_cost
1       35                88       11.61
2       26                77       13.60
3       22                82       13.14
4       20                66       13.09
5       19                67       12.89
6       15                75       12.90
7       12                58       12.54
8       12     